- APPLE:
    - Revenue: 'RevenueFromContractWithCustomerExcludingAssessedTax'
    - CapEx: 'PaymentsToAcquirePropertyPlantAndEquipment'
    - Operating Cash Flow: 'NetCashProvidedByUsedInOperatingActivities'
    - Net Income: 'NetIncomeLoss'
- MICROSOFT:
    - Revenue: 'RevenueFromContractWithCustomerExcludingAssessedTax'
    - CapEx: 'PaymentsToAcquirePropertyPlantAndEquipment'
    - Operating Cash Flow: 'NetCashProvidedByUsedInOperatingActivities'
    - Net Income: 'NetIncomeLoss'
- GOOGLE:
    - Revenue: 'Revenues'
    - CapEx: 'PaymentsToAcquirePropertyPlantAndEquipment'
    - Operating Cash Flow: 'NetCashProvidedByUsedInOperatingActivities'
    - Net Income: 'NetIncomeLoss'
- AMAZON:
    - Revenue: 'RevenueFromContractWithCustomerExcludingAssessedTax'
    - CapEx: 'PaymentsToAcquireProductiveAssets'
    - Operating Cash Flow: 'NetCashProvidedByUsedInOperatingActivities'
    - Net Income: 'NetIncomeLoss'
- META:
    - Revenue: 'RevenueFromContractWithCustomerExcludingAssessedTax'
    - CapEx: 'PaymentsToAcquirePropertyPlantAndEquipment'
    - Operating Cash Flow: 'NetCashProvidedByUsedInOperatingActivities'
    - Net Income: 'NetIncomeLoss'
- NVIDIA:
    - Revenue: 'Revenues'
    - CapEx: 'PaymentsToAcquireProductiveAssets'
    - Operating Cash Flow: 'NetCashProvidedByUsedInOperatingActivities'
    - Net Income: 'NetIncomeLoss'
- TSLA:
    - Revenue: 'Revenues'
    - CapEx: 'PaymentsToAcquirePropertyPlantAndEquipment'
    - Operating Cash Flow: 'NetCashProvidedByUsedInOperatingActivities'
    - Net Income: 'NetIncomeLoss'

In [1]:
import requests
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_rows', None)

In [3]:
CIKS = {
    'AAPL': '0000320193',
    'MSFT': '0000789019',
    'GOOGL': '0001652044',
    'AMZN': '0001018724',
    'META': '0001326801',
    'NVDA': '0001045810',
    'TSLA': '0001318605'
}

In [4]:
HEADERS = {'User-Agent': 'Ali Fazal alifazal102@gmail.com'}

def fetch_data(ticker): 
    url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{CIKS[ticker]}.json' 
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    data = r.json()
    facts = data['facts']
    accounting = facts['us-gaap']

    return accounting

In [5]:
ticker = 'META'

rev = 'RevenueFromContractWithCustomerExcludingAssessedTax'
capex = 'PaymentsToAcquirePropertyPlantAndEquipment'
ocf = 'NetCashProvidedByUsedInOperatingActivities'
ni = 'NetIncomeLoss'

accounting = fetch_data(ticker)
tag_list = [rev, capex, ocf, ni]
metrics = {tag: accounting[tag] for tag in 
           tag_list}

In [7]:
df_list = []

for metric in metrics.keys():    
    json_data = metrics[metric]['units']['USD']
    df = pd.DataFrame(json_data)
    df_list.append(df)


In [24]:
df_list = []

month_start = 1
alt_month_start = 1

for metric in metrics.keys():    
    json_data = metrics[metric]['units']['USD']
    df = pd.DataFrame(json_data)

    df = df[df['form'].isin(['10-K', '10-Q'])].copy()
    df['start'] = pd.to_datetime(df['start'])
    df['end'] = pd.to_datetime(df['end'])
    df['filed'] = pd.to_datetime(df['filed'])
    df = df.rename(columns={'val': metric})
    df = df[df['start'].dt.month.eq(month_start) | df['start'].dt.month.eq(alt_month_start)]
    
    df_latest = (df.sort_values('filed').drop_duplicates(subset='end', keep='last').sort_values('end').reset_index(drop=True))
    
    df_list.append(df_latest)

In [26]:
import pickle

with open('TSLA_list.pkl', 'wb') as f:
    pickle.dump(df_list, f)